# 🔬 Xyether Anime Upscaler - Real-Time Model Benchmark & Visual Comparison
### Compare `xyether_phase1_step_2000.pth` vs Base `net_g_89000.pth` on Google Colab GPU

In [ ]:
# @title 📦 1. Clone Engine & Setup Environment
import os, sys, subprocess

!rm -rf engine
!git clone https://github.com/Thanawanit/upscale-training.git engine
%cd engine

!pip install -q pytorch-msssim kornia huggingface_hub

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Environment Ready on {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

In [ ]:
# @title 📥 2. Download Checkpoints (Phase 5 vs Base 89k)
import urllib.request
from pathlib import Path

os.makedirs("/content/models", exist_ok=True)

# 1. Latest Phase 5 Checkpoint from Kaggle training
p5_url = "https://huggingface.co/Thanawanit/Kaggle-Backup/resolve/main/checkpoints/Phase5_Xyether_Clone/xyether_phase1_step_2000.pth"
p5_path = "/content/models/xyether_phase1_step_2000.pth"
print("📡 Downloading Phase 5 Step 2000...")
urllib.request.urlretrieve(p5_url, p5_path)

# 2. Base Pretrained net_g_89000.pth
base_url = "https://huggingface.co/Thanawanit/Kaggle-Backup/resolve/main/checkpoints/Phase4_XyetherKiller_V11/net_g_89000.pth"
base_path = "/content/models/net_g_89000.pth"
print("📡 Downloading Base net_g_89000...")
urllib.request.urlretrieve(base_url, base_path)

print(f"✅ Models downloaded successfully!")

In [ ]:
# @title 🎨 3. Prepare Test Anime Image & Load Models
import cv2, requests
import numpy as np
import torch
from archs.xyether_compact import XyetherCompactNet
from weights.transplant import perform_weight_surgery

# Prepare Transplanted Base
perform_weight_surgery("/content/models/net_g_89000.pth", "/content/models/transplanted_base.pth")

# Load Model 1: Transplanted Base (Zero-Init Head)
model_base = XyetherCompactNet().to(device)
ckpt_b = torch.load("/content/models/transplanted_base.pth", map_location=device)
model_base.load_state_dict(ckpt_b.get('params_ema', ckpt_b.get('params', ckpt_b)))
model_base.eval()

# Load Model 2: Phase 5 (Step 2000 trained on Safebooru 4K)
model_p5 = XyetherCompactNet().to(device)
ckpt_p5 = torch.load("/content/models/xyether_phase1_step_2000.pth", map_location=device)
model_p5.load_state_dict(ckpt_p5.get('params_ema', ckpt_p5.get('params', ckpt_p5)))
model_p5.eval()

# Download a sample anime art test image from Safebooru
test_url = "https://safebooru.org/images/1104/56b92bb3f4dc78be9e9661671d438140.png"
resp = requests.get(test_url, timeout=15)
with open("/content/test_anime_hr.png", "wb") as f:
    f.write(resp.content)

hr_full = cv2.imread("/content/test_anime_hr.png")
# Crop a 512x512 region with detailed line art
h, w = hr_full.shape[:2]
cy, cx = h // 2, w // 2
crop_hr = hr_full[cy-256:cy+256, cx-256:cx+256]
cv2.imwrite("/content/crop_hr.png", crop_hr)

# Downscale 2x with area interpolation to create LR input
lr_input = cv2.resize(crop_hr, (256, 256), interpolation=cv2.INTER_AREA)
cv2.imwrite("/content/lr_input.png", lr_input)

print(f"✅ Test image prepared: Input LR (256x256), Target HR (512x512)")

In [ ]:
# @title 🚀 4. Run Inference & Quantitative Benchmarks
from benchmark.xyether_evaluator import XyetherEvaluator
from IPython.display import display, HTML

# Run Quantitative Evaluator
evaluator = XyetherEvaluator(device=device)
rep_base = evaluator.evaluate_model(model_base)
rep_p5 = evaluator.evaluate_model(model_p5)

print("=== QUANTITATIVE BENCHMARK ===")
print(f"Base 89k       | Halo: {rep_base['overshoot_halo']:.2f} | Denoise: {rep_base['noise_reduction_pct']:.1f}% | Line Depth: {rep_base['line_depth']:.2f}")
print(f"Phase 5 Step 2k| Halo: {rep_p5['overshoot_halo']:.2f} | Denoise: {rep_p5['noise_reduction_pct']:.1f}% | Line Depth: {rep_p5['line_depth']:.2f}")

# Run visual inference on LR image
lr_rgb = cv2.cvtColor(lr_input, cv2.COLOR_BGR2RGB)
lr_t = torch.from_numpy(lr_rgb.transpose(2, 0, 1)).float().unsqueeze(0).to(device) / 255.0

with torch.no_grad():
    sr_base = model_base(lr_t).squeeze(0).permute(1, 2, 0).cpu().numpy()
    sr_p5 = model_p5(lr_t).squeeze(0).permute(1, 2, 0).cpu().numpy()

sr_base_bgr = (np.clip(sr_base, 0, 1) * 255.0).astype(np.uint8)[:, :, ::-1]
sr_p5_bgr = (np.clip(sr_p5, 0, 1) * 255.0).astype(np.uint8)[:, :, ::-1]
nearest_bgr = cv2.resize(lr_input, (512, 512), interpolation=cv2.INTER_NEAREST)

# Save comparison outputs
cv2.imwrite("/content/sr_base.png", sr_base_bgr)
cv2.imwrite("/content/sr_p5_2k.png", sr_p5_bgr)

# Create side-by-side zoom grid (128x128 center patch)
patch_nearest = nearest_bgr[192:320, 192:320]
patch_base = sr_base_bgr[192:320, 192:320]
patch_p5 = sr_p5_bgr[192:320, 192:320]
patch_gt = crop_hr[192:320, 192:320]

grid = np.hstack([patch_nearest, patch_base, patch_p5, patch_gt])
cv2.imwrite("/content/comparison_zoom_grid.png", grid)

import matplotlib.pyplot as plt
plt.figure(figsize=(16, 6))
plt.subplot(1, 4, 1); plt.imshow(cv2.cvtColor(patch_nearest, cv2.COLOR_BGR2RGB)); plt.title('Nearest (Input 2x)'); plt.axis('off')
plt.subplot(1, 4, 2); plt.imshow(cv2.cvtColor(patch_base, cv2.COLOR_BGR2RGB)); plt.title('Base net_g_89000'); plt.axis('off')
plt.subplot(1, 4, 3); plt.imshow(cv2.cvtColor(patch_p5, cv2.COLOR_BGR2RGB)); plt.title('Phase 5 (Step 2000)'); plt.axis('off')
plt.subplot(1, 4, 4); plt.imshow(cv2.cvtColor(patch_gt, cv2.COLOR_BGR2RGB)); plt.title('Original 4K GT'); plt.axis('off')
plt.tight_layout()
plt.show()
print('🎉 Comparison Grid Generated: /content/comparison_zoom_grid.png')